# 3 Internet comunitario en la sierra

In [23]:
from docplex.mp.model import Model
import numpy as np

mdl = Model(name='Internet comunitario en la sierra')

## Variables de Decisión

### Matriz de cobertura 
$$C = [c_{ij}] =$$

### Variable de decision
$$x_i = 
\begin{cases}
	1 & \text{si se abre la antena en el cerro $i$}
	\\
	0 & \text{si no}
\end{cases}$$

con $$i \epsilon \{ A, B, C, D, E, F \}$$

In [24]:
# Tabla 1
cobertura = [
    # 1    2  3    4    5   6   7   8    9 
    [1,   1,  0,   1,   0,  0,  0,  0,  0],  # A
    [0,   1,  1,   0,   1,  1,  0,  0,  0],  # B
    [0,   0,  0,   1,   0,  0,  1,  0,  0],  # C
    [0,   0,  0,   0,   1,  1,  1,  1,  1],  # D
    [1,   0,  1,   0,   0,  0,  0,  1,  0],  # E
    [0,   0,  0,   0,   0,  1,  1,  0,  1]   # F
]

cobertura_np = np.array(cobertura)

cobertura_np

array([[1, 1, 0, 1, 0, 0, 0, 0, 0],
       [0, 1, 1, 0, 1, 1, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 1, 1, 1, 1, 1],
       [1, 0, 1, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 1, 1, 0, 1]])

In [25]:
cerros = cobertura_np.shape[0]
localidades = cobertura_np.shape[1]

localidades

9

In [26]:
cerros_i = range(cerros)   # [0, 1, 2, 3, 4]
localidades_j = range(localidades) # [0, 1, 2, 3, 4]

In [27]:
x = mdl.binary_var_list(cerros_i, name='x')

In [28]:
x

[docplex.mp.Var(type=B,name='x_0'),
 docplex.mp.Var(type=B,name='x_1'),
 docplex.mp.Var(type=B,name='x_2'),
 docplex.mp.Var(type=B,name='x_3'),
 docplex.mp.Var(type=B,name='x_4'),
 docplex.mp.Var(type=B,name='x_5')]

#### [!!!] Correccion respecto al escrito: hacia falta una variable indicadora de cobertura
1 si la localidad esta cubierta

In [29]:
y = mdl.binary_var_list(localidades, name='y')   # Localidades cubiertas

In [30]:
y

[docplex.mp.Var(type=B,name='y_0'),
 docplex.mp.Var(type=B,name='y_1'),
 docplex.mp.Var(type=B,name='y_2'),
 docplex.mp.Var(type=B,name='y_3'),
 docplex.mp.Var(type=B,name='y_4'),
 docplex.mp.Var(type=B,name='y_5'),
 docplex.mp.Var(type=B,name='y_6'),
 docplex.mp.Var(type=B,name='y_7'),
 docplex.mp.Var(type=B,name='y_8')]

## Restricciones 

In [31]:
# costos por antena en cerro i
A = [i*1000 for i in [380, 450, 300, 520, 410, 350]]
A

[380000, 450000, 300000, 520000, 410000, 350000]

### No exceder presupuesto

In [32]:
mdl.add_constraint(
    mdl.sum(A[i] * x[i] for i in cerros_i) <= 1200000
)

docplex.mp.LinearConstraint[](380000x_0+450000x_1+300000x_2+520000x_3+410000x_4+350000x_5,LE,1200000)

#### [!!! nueva con la correccion] cobertura
$y_j=1$ si hay una antena x_i que la cubre

In [33]:
mdl.add_constraints(
    y[j] <= mdl.sum(cobertura_np[i][j] * x[i] for i in cerros_i)
    for j in localidades_j
)

[docplex.mp.LinearConstraint[](y_0,LE,x_0+x_4),
 docplex.mp.LinearConstraint[](y_1,LE,x_0+x_1),
 docplex.mp.LinearConstraint[](y_2,LE,x_1+x_4),
 docplex.mp.LinearConstraint[](y_3,LE,x_0+x_2),
 docplex.mp.LinearConstraint[](y_4,LE,x_1+x_3),
 docplex.mp.LinearConstraint[](y_5,LE,x_1+x_3+x_5),
 docplex.mp.LinearConstraint[](y_6,LE,x_2+x_3+x_5),
 docplex.mp.LinearConstraint[](y_7,LE,x_3+x_4),
 docplex.mp.LinearConstraint[](y_8,LE,x_3+x_5)]

## Funcion objetivo

#### Maximizar habitantes cubiertos

In [34]:
# habitantes por localidad j
B = [850, 1200, 430, 2100, 670, 980, 1530, 390, 760]

[!!!] $y_j$ implementada

In [35]:
mdl.maximize(
    mdl.sum(B[j] * y[j] for j in localidades_j)
)

# Resolver e imprimir solucion
solution = mdl.solve(log_output=True)


print(solution)

Version identifier: 22.2.0.0 | 2026-05-04 | 5dff98e58
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 0.000000 after 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
MIP Presolve modified 1 coefficients.
Reduced MIP has 10 rows, 15 columns, and 35 nonzeros.
Reduced MIP has 15 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.01 sec. (0.02 ticks)
Probing time = 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 10 rows, 15 columns, and 35 nonzeros.
Reduced MIP has 15 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.02 ticks)
Probing time = 0.00 sec. (0.00 ticks)
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 4 threads.
Root relaxation solution time = 0.00 sec. (0.02 ticks)

        Nodes                                         Cuts/
   Node  Left     Objective  IInf  Best Integer    Best Bound    ItCnt

### b) Al menos dos antenas cubriendo $L_4$ y $L_7$

Añadimos restriccion

In [36]:
mdl.add_constraints(
    mdl.sum(cobertura_np[i][j] * x[i] for i in cerros_i) >= 2
    for j in [4-1, 7-1]
)

[docplex.mp.LinearConstraint[](x_0+x_2,GE,2),
 docplex.mp.LinearConstraint[](x_2+x_3+x_5,GE,2)]

In [37]:
# Resolver e imprimir solucion
solution = mdl.solve(log_output=True)


print(solution)

Version identifier: 22.2.0.0 | 2026-05-04 | 5dff98e58
CPXPARAM_Read_DataCheck                          1
Retaining values of one MIP start for possible repair.
Tried aggregator 1 time.
MIP Presolve eliminated 12 rows and 15 columns.
MIP Presolve modified 4 coefficients.
All rows and columns eliminated.
Presolve time = 0.01 sec. (0.02 ticks)

Root node processing (before b&c):
  Real time             =    0.02 sec. (0.03 ticks)
Parallel b&c, 4 threads:
  Real time             =    0.00 sec. (0.00 ticks)
  Sync time (average)   =    0.00 sec.
  Wait time (average)   =    0.00 sec.
                          ------------
Total (root+branch&cut) =    0.02 sec. (0.03 ticks)
solution for: Internet comunitario en la sierra
objective: 8480
status: OPTIMAL_SOLUTION(2)
x_0=1
x_2=1
x_3=1
y_0=1
y_1=1
y_3=1
y_4=1
y_5=1
y_6=1
y_7=1
y_8=1

